# Energy Demand Forecaster: Phase 3 - Naive & Classical Benchmarks

**Objective**: Establish rigorous benchmarks on the unseen test set before training complex neural networks. Any neural model (RNN, LSTM, GRU) must honestly beat these baselines to justify computational overhead.

**Syllabus Topic Mapping**:
- Regression evaluation metrics: MSE, MAE, RMSE, $R^2$, MAPE
- Baseline modeling and the importance of naive persistence in sequential data

In [ ]:
from pathlib import Path

import pandas as pd

from src.models.baselines import evaluate_all_baselines

## 1. Load Processed Clean Dataset & Chronological Splits

In [ ]:
proc_path = Path('../data/processed/spain_hourly_clean.parquet')
df = pd.read_parquet(proc_path)

train_split = '2017-10-18 23:00:00'
test_split = '2018-05-26 00:00:00'

df_train = df.loc[:train_split]
df_test = df.loc[test_split:]

print(f"Train observations: {len(df_train)}")
print(f"Test observations: {len(df_test)}")

## 2. Compute Benchmark Metrics
- **Naive (1h)**: $\hat{y}_t = y_{t-1}$
- **Seasonal Naive (24h)**: $\hat{y}_t = y_{t-24}$
- **Seasonal Naive (168h)**: $\hat{y}_t = y_{t-168}$
- **Linear Regression on Lags**: Ridge fitted on $y_{t-24}, y_{t-48}, y_{t-168}$ and rolling stats
- **SARIMA**: $(1,1,1) \times (1,0,0)_{24}$

In [ ]:
baseline_results = evaluate_all_baselines(df_train, df_test, target_col='total_load_actual')

df_res = pd.DataFrame(baseline_results).T
df_res[['rmse', 'mae', 'mape', 'r2']].round({'rmse': 2, 'mae': 2, 'mape': 2, 'r2': 4})

## 3. Analysis & Key Insights
- **Seasonal Naive (24h)** achieves an RMSE of ~1,488 MW and MAPE of 3.95% without any parameters, demonstrating the power of daily circadian rhythm.
- **Ridge Regression** on historical lags captures both 24h and 168h periodicities, dropping RMSE to ~995 MW.
- Any neural architecture must achieve an RMSE significantly below 1,000 MW to demonstrate genuine non-linear value.